In [ ]:
#| default_exp features.fsc_regions

In [ ]:
#| export
from __future__ import annotations
import pandas as pd
import structlog
from kreview.eval_engine import FeatureEvaluator

log = structlog.get_logger()


In [ ]:
#| export
class FSCRegionsEvaluator(FeatureEvaluator):
    """Extracts variance of fragment ratios partitioned by specific targeted regions."""
    
    name = "FSC_regions"
    source_file = ".FSC.regions.parquet"
    tier = 1
    category = "fragmentation"

    def extract(self, df: pd.DataFrame) -> dict[str, float]:
        extracted = {}
        try:
            if df.empty:
                return extracted
            
            cols = set(df.columns)
            ratios = ["core_short_ratio", "mono_nucl_ratio"]
            
            for r in ratios:
                if r in cols:
                    extracted[f"regional_{r}_mean"] = float(df[r].mean())
                    extracted[f"regional_{r}_std"] = float(df[r].std())
            
            # Aggregate per gene if region_mapped
            if "gene" in cols and "core_short_ratio" in cols:
                gene_means = df.groupby("gene")["core_short_ratio"].mean()
                extracted["core_short_ratio_variance_across_genes"] = float(gene_means.std())

            return extracted
        except Exception as e:
            log.warning("fsc_regions_extraction_failed", error=str(e))
            return {}


In [ ]:
#| test
evaluator = FSCRegionsEvaluator()
df_test = pd.DataFrame([{"gene": "TP53", "core_short_ratio": 0.2}, {"gene": "EGFR", "core_short_ratio": 0.4}])
res = evaluator.extract(df_test)
assert "regional_core_short_ratio_mean" in res
assert "core_short_ratio_variance_across_genes" in res
